# #9.4 Handoffs — 직접 돌려보기

**목적**: triage 에이전트가 손님 요청을 보고 **알맞은 전문 에이전트로 넘기는(handoff)** 걸 눈으로 확인.

**어제 퀴즈 복습**: handoff = 다른 에이전트로 **완전 이관**(대화가 그쪽으로 넘어감). 정체는 SDK가 자동 등록하는 **`transfer_to_<agent>` tool**. (vs as-tool = triage가 물어보고 결과만 전달)

In [1]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner

load_dotenv()
print("키 있음 ✅" if os.getenv("OPENAI_API_KEY") else "키 없음 ❌")

키 있음 ✅


## 1) 전문 에이전트 4종 + Triage

- 전문가 3명: Menu / Order / Reservation
- `handoff_description` = triage가 "누구한테 넘길지" 판단할 때 보는 설명
- triage `handoffs=[...]` 에 전문가들을 등록 → triage는 **직접 답하지 않고 분류만**

In [2]:
menu_agent = Agent(
    name="Menu Agent",
    handoff_description="메뉴, 재료, 알레르기 관련 질문 담당",
    instructions="너는 레스토랑 메뉴 전문가야. 메뉴/재료/알레르기 질문에 한국어 존댓말로 친절히 답해.",
)
order_agent = Agent(
    name="Order Agent",
    handoff_description="주문 받기와 주문 확인 담당",
    instructions="너는 주문 담당이야. 손님 주문을 받고 내용을 확인해줘. 한국어 존댓말.",
)
reservation_agent = Agent(
    name="Reservation Agent",
    handoff_description="테이블 예약 처리 담당",
    instructions="너는 예약 담당이야. 인원수, 날짜, 시간을 물어 테이블 예약을 도와줘. 한국어 존댓말.",
)

triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "너는 식당 안내 데스크야. 손님 요청을 보고 메뉴/주문/예약 중 알맞은 전문가에게 넘겨(handoff). "
        "직접 답하려 하지 말고 분류해서 전달해."
    ),
    handoffs=[menu_agent, order_agent, reservation_agent],
)
print("에이전트 준비됨. triage handoffs:", [a.name for a in triage_agent.handoffs])

에이전트 준비됨. triage handoffs: ['Menu Agent', 'Order Agent', 'Reservation Agent']


## 2) 예약 요청 → 어디로 넘어가나?

triage 에 예약 요청을 던진다. 끝나면 **`result.last_agent`** 가 누구인지 본다 — handoff 됐으면 **Reservation Agent**로 바뀌어 있어야 한다.

In [3]:
result = await Runner.run(triage_agent, "테이블 예약하고 싶어. 금요일 저녁 4명이야.")
print("💬 응답:", result.final_output)
print("\n👤 마지막 담당 에이전트:", result.last_agent.name)  # ← Reservation Agent 면 handoff 성공

💬 응답: 네, 예약 도와드리겠습니다.  
금요일 저녁 4명으로 예약 원하시는군요.

예약을 위해 **정확한 날짜와 시간**을 알려주시겠어요?  
예: 3월 15일 저녁 7시 30분

👤 마지막 담당 에이전트: Reservation Agent


## 3) handoff 흔적 보기

`result.new_items` 에 이번 실행에서 생긴 것들이 들어있다. handoff가 일어났으면 **`transfer_to_...`** 호출/출력 아이템이 보인다. (이게 #9.5에서 UI에 "예약 담당에게 연결합니다"로 표시하는 그 이벤트)

In [4]:
for item in result.new_items:
    print(item.__class__.__name__, "|", getattr(item, "type", ""))

HandoffCallItem | handoff_call_item
HandoffOutputItem | handoff_output_item
MessageOutputItem | message_output_item


## 4) 메뉴 질문 → Menu Agent 로

같은 triage에 이번엔 메뉴 질문. 담당이 **Menu Agent**로 바뀌면 = triage가 요청 내용에 따라 알맞게 라우팅한다는 증거.

In [5]:
result2 = await Runner.run(triage_agent, "채식 메뉴 있어? 견과류 알레르기도 있는데 괜찮은 거 알려줘.")
print("💬 응답:", result2.final_output)
print("\n👤 마지막 담당 에이전트:", result2.last_agent.name)  # ← Menu Agent

💬 응답: 네, 채식 메뉴와 견과류 알레르기까지 고려해서 안내드릴 수 있습니다.

가능하면 아래 3가지를 알려주세요:
1. **현재 메뉴판 사진** 또는 메뉴 목록  
2. **완전 채식(비건)**인지, **달걀/유제품은 괜찮은지**  
3. **견과류 종류**(땅콩, 호두, 아몬드 등) 중 특히 피해야 하는 것

메뉴를 보내주시면  
- **채식 가능 메뉴**
- **견과류 제외 가능한 메뉴**
- **주의가 필요한 메뉴(소스/토핑 포함)**  
이렇게 구분해서 바로 알려드릴게요.

👤 마지막 담당 에이전트: Menu Agent


## 5) 정리 + 앱에서 주의할 것 (#9.5 버그)

- **handoff vs as-tool**: handoff = 대화가 전문가에게 **완전 이관**(`result.last_agent`가 바뀜) / as-tool = triage가 내부적으로 물어보고 결과만 전달(유저는 계속 triage와).
- **⚠️ #9.5 멀티턴 버그**: 채팅 앱에서 매 메시지를 **항상 triage로** 시작하면, 예약 담당에게 넘어간 뒤 다음 메시지가 **또 triage로** 돌아가버린다.
  - 해결: 직전 `result.last_agent`를 **`st.session_state`에 저장**해두고, 다음 턴엔 그 에이전트로 `Runner.run` → 대화가 이어짐. (예약 도중 "아 메뉴 먼저" 하면 다시 triage/menu로 넘어가도록 처리)
- 다음 단계(B): 이 4 에이전트를 Streamlit 앱으로 + handoff 표시("🔀 예약 담당에게 연결합니다...") + active agent 캐싱.